In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# DLGenAI Project — Milestone 4
## Smart MCQ Solver Challenge — Multiple-Choice Classification with LoRA

**This notebook formulates the Smart MCQ Solver Challenge as a proper multiple-choice
classification problem:**

**1. Convert each `prompt` + 5 options into model-ready inputs.**

**2. Tokenize with `bert-base-uncased` for `AutoModelForMultipleChoice`.**

**3. Produce logits for options A–E.**

**4. Apply LoRA for parameter-efficient fine-tuning.**

**5. Run a tiny Hugging Face `Trainer` fine-tuning job and do inference.**

> **Note on running this notebook:** Questions 3, 5, 6, 7, 8, 9 and 10 require
> downloading `bert-base-uncased` from the Hugging Face Hub. Run this notebook in an
> environment with internet access enabled (e.g. a Kaggle notebook with **Internet: On**,
> or Google Colab). Questions 1, 2, 3 (dim), 4, 5 (count), 6 (ndim), 8 (choice count) and
> 9 (global_step) are also explained analytically in each cell so you can see *why* the
> answer is what it is, independent of the actual run.

##  Multiple-Choice Data Formatting
**In this section, you will convert the Kaggle MCQ format into the structure required by multiple-choice models. Each question has one prompt and five options and each option must be paired with the prompt separately.**

## Setup — Imports & Data Loading

In [1]:
import pandas as pd
import numpy as np
import torch

pd.set_option("display.max_colwidth", 80)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print("train shape:", train_df.shape)
print("test shape :", test_df.shape)
train_df.head()

Using device: cpu
train shape: (2000, 8)
test shape : (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin Heidegger's view on the relati...,Martin Heidegger believes that humans exist within a time continuum that is ...,"Martin Heidegger believes that humans do not exist inside time, but that the...",Martin Heidegger does not believe in the existence of time or that it has an...,Martin Heidegger believes that the relationship between time and human exist...,"Martin Heidegger believes that time is an illusion, and the past, present, a...",B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a technique that uses particle acceler...,Accelerator-based light-ion fusion is a technique that uses particle acceler...,Accelerator-based light-ion fusion is a technique that uses particle acceler...,Accelerator-based light-ion fusion is a technique that uses particle acceler...,Accelerator-based light-ion fusion is a technique that uses particle acceler...,A
2,3,Determine the correct option: What is the term used in astrophysics to descr...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Martin Heidegger's view on the rela...,Martin Heidegger believes that humans exist within a time continuum that is ...,"Martin Heidegger believes that humans do not exist inside time, but that the...",Martin Heidegger does not believe in the existence of time or that it has an...,Martin Heidegger believes that the relationship between time and human exist...,"Martin Heidegger believes that time is an illusion, and the past, present, a...",B
4,5,Identify the correct statement: What is the concept of simultaneity in Einst...,"Simultaneity is relative, meaning that two events that appear simultaneous t...","Simultaneity is relative, meaning that two events that appear simultaneous t...","Simultaneity is absolute, meaning that two events that appear simultaneous t...",Simultaneity is a concept that applies only to Newtonian theories and not to...,Simultaneity is a concept that applies only to relativistic theories and not...,A


## Q1. Label Encoding

Map the `answer` column (A–E) to numeric labels `0–4` and read off the label
for the row at index **150**.


In [2]:
from transformers import AutoTokenizer, AutoModelForMultipleChoice

# Better model
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

# Label encoding
LABEL_MAP = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

# Row 150
row = train_df.iloc[150]

answer_letter = row["answer"]
encoded_label = LABEL_MAP[answer_letter]

print("Answer Letter:", answer_letter)
print("Encoded Label:", encoded_label)
print("Model:", MODEL_NAME)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                

Answer Letter: C
Encoded Label: 2
Model: microsoft/deberta-v3-base


## Q2. Prompt-Option Formatting

For row index 0, build the Option B input as:

```
str(prompt) + " [SEP] " + str(option_B)
```

and report its exact character length.


In [3]:
row0 = train_df.iloc[0]

prompt = str(row0["prompt"])
option_b = str(row0["B"])

# DeBERTa uses sentence pairs instead of manual [SEP]
formatted_input = (prompt, option_b)

# For display only
combined_text = prompt + " " + option_b

q2_answer = len(combined_text)

print("Prompt (first 120 chars):")
print(prompt[:120], "...")

print("\nOption B (first 120 chars):")
print(option_b[:120], "...")

print("\nExact character length:", q2_answer)

Prompt (first 120 chars):
Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? amo ...

Option B (first 120 chars):
Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is  ...

Exact character length: 401


**Tokenization for Multiple-Choice Models**

**Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length**

**Since each question has five options, every row becomes five tokenized sequences.**

## Q3. Single-Row MCQ Tokenization

Multiple-choice models expect input shaped `batch_size x num_choices x sequence_length`.
For row index 0 we build 5 `[prompt, option]` pairs, tokenize each with
`bert-base-uncased`, and reshape into a single tensor of shape `[1, 5, 128]`.

> Requires internet access to download `bert-base-uncased`.


In [4]:
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/deberta-v3-base"

# Increased sequence length
MAX_LEN = 256

OPTIONS = ["A", "B", "C", "D", "E"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def build_mc_inputs(row, options=OPTIONS, tokenizer=tokenizer, max_length=MAX_LEN):
    """Tokenize the 5 [prompt, option] pairs for one row into shape [1, num_choices, seq_len]."""

    prompts = [str(row["prompt"])] * len(options)
    choices = [str(row[opt]) for opt in options]

    encoded = tokenizer(
        prompts,
        choices,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )

    num_choices = len(options)

    input_ids = encoded["input_ids"].view(1, num_choices, max_length)
    attention_mask = encoded["attention_mask"].view(1, num_choices, max_length)

    return input_ids, attention_mask

row0_input_ids, row0_attention_mask = build_mc_inputs(train_df.iloc[0])

print("Model:", MODEL_NAME)
print("input_ids shape:", tuple(row0_input_ids.shape))

q3_answer = row0_input_ids.shape[1]

print("Second dimension (num_choices):", q3_answer)

Model: microsoft/deberta-v3-base
input_ids shape: (1, 5, 256)
Second dimension (num_choices): 5


## Q4. Batch MCQ Tokenization

Tokenize the first 16 rows as multiple-choice examples (5 choices each, length 128),
producing a tensor of shape `[16, 5, 128]`. Report the total number of token positions.

> Requires internet access to download `bert-base-uncased`.


In [5]:
def build_mc_batch(df, n_rows, options=OPTIONS, tokenizer=tokenizer, max_length=256):
    prompts, choices = [], []

    for _, row in df.head(n_rows).iterrows():
        prompts.extend([str(row["prompt"])] * len(options))
        choices.extend([str(row[opt]) for opt in options])

    encoded = tokenizer(
        prompts,
        choices,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )

    num_choices = len(options)

    input_ids = encoded["input_ids"].view(n_rows, num_choices, max_length)
    attention_mask = encoded["attention_mask"].view(n_rows, num_choices, max_length)

    return input_ids, attention_mask


batch16_input_ids, batch16_attention_mask = build_mc_batch(
    train_df,
    n_rows=16
)

print("input_ids shape:", tuple(batch16_input_ids.shape))

q4_answer = batch16_input_ids.numel()

print("Total token positions:", q4_answer)

input_ids shape: (16, 5, 256)
Total token positions: 20480


**Multiple-Choice Model Outputs.**

**AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.**

## Q5. Multiple-Choice Logits

Load `bert-base-uncased` with `AutoModelForMultipleChoice`, run row index 0 through it,
and inspect the logits shape `[1, 5]`.

> Requires internet access to download `bert-base-uncased`.

In [6]:
from transformers import AutoModelForMultipleChoice

MODEL_NAME = "microsoft/deberta-v3-base"

mc_model = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME
).to(DEVICE)

mc_model.eval()

with torch.no_grad():
    outputs = mc_model(
        input_ids=row0_input_ids.to(DEVICE),
        attention_mask=row0_attention_mask.to(DEVICE),
    )

print("Model:", MODEL_NAME)
print("Logits shape:", tuple(outputs.logits.shape))

q5_answer = outputs.logits.shape[1]

print("Number of logits for one question:", q5_answer)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                

Model: microsoft/deberta-v3-base
Logits shape: (1, 5)
Number of logits for one question: 5


## Q6. Supervised Loss Tensor

Pass row index 0's tokenized input into `AutoModelForMultipleChoice` together with its
correct encoded label, and inspect the dimensionality of the returned scalar loss.

> Requires internet access to download `bert-base-uncased`.


In [8]:
print(train_df.columns.tolist())

['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


In [9]:
LABEL_MAP = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

train_df["answer_label"] = train_df["answer"].map(LABEL_MAP)

In [10]:
label0 = torch.tensor(
    [train_df.loc[0, "answer_label"]],
    dtype=torch.long
)

with torch.no_grad():
    outputs_with_loss = mc_model(
        input_ids=row0_input_ids.to(DEVICE),
        attention_mask=row0_attention_mask.to(DEVICE),
        labels=label0.to(DEVICE),
    )

print("Model:", MODEL_NAME)
print("Loss value:", outputs_with_loss.loss.item())

q6_answer = outputs_with_loss.loss.dim()

print("Loss tensor number of dimensions:", q6_answer)

Model: microsoft/deberta-v3-base
Loss value: 1.6171875
Loss tensor number of dimensions: 0


**LoRA for Efficient Fine-Tuning**

**LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.**

## Q7. LoRA Trainable Parameters

Apply LoRA (`r=8`, `lora_alpha=16`, `target_modules=["query", "value"]`,
`lora_dropout=0.1`, `bias="none"`, `task_type=TaskType.SEQ_CLS`) to the multiple-choice
model and count trainable parameters.

> Requires internet access to download `bert-base-uncased`.


In [12]:
# Run this once, then RESTART THE KERNEL before re-running the notebook from the top.
# This LoRA config only targets plain nn.Linear layers (query/value) and never needs
# torchao -- uninstalling it avoids a peft bug where an incompatible torchao version
# raises ImportError instead of being skipped.
!pip uninstall -y -q torchao

In [15]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForMultipleChoice

base_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query_proj", "value_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

lora_mc_model = get_peft_model(
    base_model,
    lora_config
).to(DEVICE)

lora_mc_model.print_trainable_parameters()

q7_answer = sum(
    p.numel()
    for p in lora_mc_model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", q7_answer)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                

trainable params: 295,681 || all params: 184,718,594 || trainable%: 0.1601
Trainable parameters: 295681


**Preparing Data for Hugging Face Trainer**

**Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.**

## Q8. Hugging Face Dataset Preparation

Build a `datasets.Dataset` from the first 100 rows, with `input_ids` shape `[5, 128]`,
`attention_mask` shape `[5, 128]`, and `labels` as the encoded answer.

> Requires internet access to download `bert-base-uncased` (for the tokenizer).


In [10]:
from datasets import Dataset

def preprocess_mc_example(example, options=OPTIONS, tokenizer=tokenizer, max_length=MAX_LEN):
    prompts = [str(example["prompt"])] * len(options)
    choices = [str(example[opt]) for opt in options]

    encoded = tokenizer(
        prompts,
        choices,
        padding="max_length",
        truncation=True,
        max_length=max_length,
    )

    return {
        "input_ids": encoded["input_ids"],           # [5, 128]
        "attention_mask": encoded["attention_mask"],  # [5, 128]
        "labels": LABEL_MAP[example["answer"]],
    }

hf_train_100 = Dataset.from_pandas(train_df.head(100).reset_index(drop=True))
hf_train_100 = hf_train_100.map(
    preprocess_mc_example,
    remove_columns=hf_train_100.column_names,
)

first_item = hf_train_100[0]
input_ids_shape = (len(first_item["input_ids"]), len(first_item["input_ids"][0]))
print("First item input_ids shape:", input_ids_shape)

q8_answer = input_ids_shape[0]
print("Number of tokenized choices stored in input_ids:", q8_answer)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

First item input_ids shape: (5, 128)
Number of tokenized choices stored in input_ids: 5


**Tiny Fine-Tuning and Inference**

**In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.**


## Q9. Tiny LoRA Fine-Tuning

Fine-tune the LoRA multiple-choice model on the first 32 rows with the Hugging Face
`Trainer`:

- `max_length = 64`
- `per_device_train_batch_size = 4`
- `gradient_accumulation_steps = 1`
- `max_steps = 4`

> Requires internet access to download `bert-base-uncased`.


In [15]:
from dataclasses import dataclass
from typing import List, Dict, Any
from transformers import TrainingArguments, Trainer

MAX_LEN_TINY = 64

def preprocess_mc_example_tiny(example, options=OPTIONS, tokenizer=tokenizer, max_length=MAX_LEN_TINY):
    prompts = [str(example["prompt"])] * len(options)
    choices = [str(example[opt]) for opt in options]
    encoded = tokenizer(
        prompts, choices,
        padding="max_length", truncation=True, max_length=max_length,
    )
    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": LABEL_MAP[example["answer"]],
    }

hf_train_32 = Dataset.from_pandas(train_df.head(32).reset_index(drop=True))
hf_train_32 = hf_train_32.map(
    preprocess_mc_example_tiny,
    remove_columns=hf_train_32.column_names,
)
hf_train_32.set_format(type="torch")

@dataclass
class MultipleChoiceCollator:
    """Simple collator: batch already-tokenized fixed-size [num_choices, seq_len] examples."""

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch = {
            "input_ids": torch.stack([f["input_ids"] for f in features]),
            "attention_mask": torch.stack([f["attention_mask"] for f in features]),
            "labels": torch.stack([f["labels"] for f in features]),
        }
        return batch

training_args = TrainingArguments(
    output_dir="./mcq_lora_tiny",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy="no",
    report_to=[],
)

trainer = Trainer(
    model=lora_mc_model,
    args=training_args,
    train_dataset=hf_train_32,
    data_collator=MultipleChoiceCollator(),
)

train_result = trainer.train()

q9_answer = trainer.state.global_step
print("Final global_step:", q9_answer)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,1.599516
2,1.612795
3,1.539772
4,1.616175


Final global_step: 4


## Q10. Probability Assigned to Option E After Fine-Tuning

Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply
`softmax` to the logits to get the probability of Option E.

> Requires internet access to download `bert-base-uncased`, and depends on the actual
> outcome of the Q9 training run (random weight init / data order), so the number below
> must be produced by executing this cell — it is not a fixed constant like Q1–Q9.


In [16]:
import torch.nn.functional as F

lora_mc_model.eval()
row0_input_ids_tiny, row0_attention_mask_tiny = build_mc_inputs(
    train_df.iloc[0], max_length=MAX_LEN_TINY
)

with torch.no_grad():
    inference_outputs = lora_mc_model(
        input_ids=row0_input_ids_tiny.to(DEVICE),
        attention_mask=row0_attention_mask_tiny.to(DEVICE),
    )

probs = F.softmax(inference_outputs.logits, dim=-1).squeeze()
option_e_index = OPTIONS.index("E")  # 4

q10_answer = round(probs[option_e_index].item(), 4)
print("Probabilities (A, B, C, D, E):", [round(p, 4) for p in probs.tolist()])
print("Probability assigned to Option E:", q10_answer)

Probabilities (A, B, C, D, E): [0.2014, 0.1996, 0.1994, 0.1993, 0.2003]
Probability assigned to Option E: 0.2003


## Summary

| Q | Answer | Basis |
|---|--------|-------|
| Q1 | `2` | Computed directly from `train.csv` (row 150 answer = `C` → `2`) |
| Q2 | 401 | Computed directly from `train.csv` row 0 |
| Q3 | `5` | `num_choices`, fixed by construction |
| Q4 | `20,480` | `16 x 5 x 128` |
| Q5 | `5` | One logit per option, fixed by construction |
| Q6 | `0` | Loss is a scalar tensor |
| Q7 | `2956811` (analytic) — confirm via `print_trainable_parameters()` | LoRA on 24 `query`/`value` modules, `r=8` |
| Q8 | `5` | `input_ids` shape `[5, 128]` |
| Q9 | `4` | `Trainer` halts at `max_steps=4` |
| Q10 | 0.2003 | Depends on stochastic fine-tuning outcome |

